Installing the dependencies - langchain framework and flan-T5 model

In [2]:
!pip install -q --upgrade \
langchain \
langchain-community \
transformers \
sentence-transformers \
faiss-cpu \
pypdf \
requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 93.4 MB/s eta 0:00:00


In [12]:
#Imports

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

In [4]:
# Sample data
with open("company_data.txt", "w") as f:
    f.write("""
    Kect Management Services is a recruitment company in Sri Lanka.
    We specialize in foreign employment opportunities.
    We provide interview coordination, documentation support, and visa processing.
    Candidates must submit CV, passport copy, and educational certificates.
    """)

# Load and split the above created document
loader = TextLoader("company_data.txt")
documents = loader.load()

text_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

Embedding with hugging face model

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

/tmp/ipykernel_8599/4145769993.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Store vector in FAISS Database

In [7]:
db = FAISS.from_documents(docs, embeddings)
retriever = db.as_retriever()

LLM- Free

In [14]:
pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=200
)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'Diff

RAG

In [15]:
def ask_bot(question):
    relevant_docs = retriever.get_relevant_documents(question)

    context = " ".join([doc.page_content for doc in relevant_docs])

    prompt = f"""
    Answer ONLY from the context below.
    If the answer is not in the context, say "I don't know".

    Context:
    {context}

    Question: {question}
    """

    result = pipe(prompt)
    return result[0]['generated_text']

Testing

In [16]:
print(ask_bot("What does Kect Management Services do?"))
print(ask_bot("What documents are required?"))

/tmp/ipykernel_8599/71033579.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_docs = retriever.get_relevant_documents(question)



    Answer ONLY from the context below.
    If the answer is not in the context, say "I don't know".

    Context:
    Kect Management Services is a recruitment company in Sri Lanka.
    We specialize in foreign employment opportunities.
    We provide interview coordination, documentation support, and visa processing.
    Candidates must submit CV, passport copy, and educational certificates.

    Question: What does Kect Management Services do?
    

    Answer ONLY from the context below.
    If the answer is not in the context, say "I don't know".

    Context:
    Kect Management Services is a recruitment company in Sri Lanka.
    We specialize in foreign employment opportunities.
    We provide interview coordination, documentation support, and visa processing.
    Candidates must submit CV, passport copy, and educational certificates.

    Question: What documents are required?
    


In [18]:
while True:
    q = input("\nAsk something (or type exit): ")
    if q.lower() == "exit":
        break
    print("Bot:", ask_bot(q))


Ask something (or type exit): hi
Bot: 
    Answer ONLY from the context below.
    If the answer is not in the context, say "I don't know".

    Context:
    Kect Management Services is a recruitment company in Sri Lanka.
    We specialize in foreign employment opportunities.
    We provide interview coordination, documentation support, and visa processing.
    Candidates must submit CV, passport copy, and educational certificates.

    Question: hi
    

Ask something (or type exit): exit
